In [1]:
# table3_rebuild_meanpm.py
# 목적:
# - 기존 Table 3(Seed=42 단일 결과 기반 "best model")을
#   "여러 seed/run 평균 ± 표준편차" 형태로 재작성
# - best model 선택 기준은 기존과 동일하게 AUPRC (평균) 최대
#
# 입력 파일(모두 현재 폴더 ./):
#   results_O_1.csv ... results_O_5.csv
#   results_F_1.csv ... results_F_5.csv
#   results_OF_1.csv ... results_OF_5.csv
#
# 출력:
#   table3_meanpm.csv
#
# 가정:
# - 각 results_*.csv에는 최소 컬럼:
#   SET, DAG, MODEL, K_EDGE, N_FEAT, FEATURE_KEY, AUROC, AUPRC, F1, Brier, ECE
# - O의 경우 DAG는 있어도 무시 가능(원래 구조학습 없음)

import os
import re
import glob
import pandas as pd

METRICS = ["AUROC", "AUPRC", "F1", "Brier", "ECE"]

def load_results(pattern: str) -> pd.DataFrame:
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No files matched: {pattern}")

    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        # seed/run id를 파일명에서 추출: results_F_3.csv -> 3
        m = re.search(r"_(\d+)\.csv$", os.path.basename(p))
        df["SEED_RUN"] = int(m.group(1)) if m else None
        df["SOURCE_FILE"] = os.path.basename(p)
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)

    # 필수 컬럼 체크
    required = ["SET", "DAG", "MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"] + METRICS
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    return out

def fmt(mean: float, std: float, digits: int) -> str:
    return f"{mean:.{digits}f} ± {std:.{digits}f}"

def build_table3_meanpm(df_all: pd.DataFrame) -> pd.DataFrame:
    # 1) F/OF: (SET, DAG)별로 후보 (MODEL, K_EDGE, N_FEAT, FEATURE_KEY)들의
    #    seed-run 평균 AUPRC가 최대인 1개를 best로 선택
    group_cols = ["SET", "DAG", "MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"]
    agg = (
        df_all.groupby(group_cols)[METRICS]
             .agg(["mean", "std", "count"])
             .reset_index()
    )
    # flatten columns
    agg.columns = [
        "_".join([x for x in col if x]) if isinstance(col, tuple) else col
        for col in agg.columns
    ]
    # rename key cols back
    for c in ["SET", "DAG", "MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"]:
        if c + "_" in agg.columns:
            agg.rename(columns={c + "_": c}, inplace=True)

    # best config per (SET, DAG) using AUPRC_mean
    agg_sorted = agg.sort_values(["SET", "DAG", "AUPRC_mean"], ascending=[True, True, False])
    best_F_OF = agg_sorted.groupby(["SET", "DAG"], as_index=False).head(1)

    # 2) O: 원래 Table 3는 O에서 "전체 모델 중 AUPRC 최고" 1개를 선택 (DAG 무의미)
    o = df_all[df_all["SET"] == "O"].copy()
    if o.empty:
        raise ValueError("No rows for SET='O' found.")

    o_group_cols = ["MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"]
    o_agg = (
        o.groupby(o_group_cols)[METRICS]
         .agg(["mean", "std", "count"])
         .reset_index()
    )
    o_agg.columns = [
        "_".join([x for x in col if x]) if isinstance(col, tuple) else col
        for col in o_agg.columns
    ]
    o_best = o_agg.sort_values("AUPRC_mean", ascending=False).head(1)
    if o_best.empty:
        raise ValueError("Could not select best model for O.")

    # 3) 표 형태로 정리(Mean ± SD)
    rows = []

    # O row
    rows.append({
        "Feature Set": "O",
        "DAG Method": "–",
        "Best Model": o_best["MODEL"].iloc[0],
        "K_EDGE": int(o_best["K_EDGE"].iloc[0]),
        "N_FEAT": int(o_best["N_FEAT"].iloc[0]),
        "AUROC (Mean ± SD)": fmt(o_best["AUROC_mean"].iloc[0], o_best["AUROC_std"].iloc[0], 3),
        "AUPRC (Mean ± SD)": fmt(o_best["AUPRC_mean"].iloc[0], o_best["AUPRC_std"].iloc[0], 3),
        "F1-Score (Mean ± SD)": fmt(o_best["F1_mean"].iloc[0], o_best["F1_std"].iloc[0], 3),
        "Brier (Mean ± SD)": fmt(o_best["Brier_mean"].iloc[0], o_best["Brier_std"].iloc[0], 5),
        "ECE (Mean ± SD)": fmt(o_best["ECE_mean"].iloc[0], o_best["ECE_std"].iloc[0], 5),
    })

    # F / OF rows
    for set_name in ["F", "OF"]:
        tmp = best_F_OF[best_F_OF["SET"] == set_name].copy()
        if tmp.empty:
            continue
        tmp = tmp.sort_values("DAG")
        for _, r in tmp.iterrows():
            rows.append({
                "Feature Set": set_name,
                "DAG Method": r["DAG"],
                "Best Model": r["MODEL"],
                "K_EDGE": int(r["K_EDGE"]),
                "N_FEAT": int(r["N_FEAT"]),
                "AUROC (Mean ± SD)": fmt(r["AUROC_mean"], r["AUROC_std"], 3),
                "AUPRC (Mean ± SD)": fmt(r["AUPRC_mean"], r["AUPRC_std"], 3),
                "F1-Score (Mean ± SD)": fmt(r["F1_mean"], r["F1_std"], 3),
                "Brier (Mean ± SD)": fmt(r["Brier_mean"], r["Brier_std"], 5),
                "ECE (Mean ± SD)": fmt(r["ECE_mean"], r["ECE_std"], 5),
            })

    return pd.DataFrame(rows)

def main():
    # 모든 result 파일 로드
    df_O  = load_results("./results_O_*.csv")
    df_F  = load_results("./results_F_*.csv")
    df_OF = load_results("./results_OF_*.csv")

    df_all = pd.concat([df_O, df_F, df_OF], ignore_index=True)

    # Table 3 (mean ± std) 생성
    table3 = build_table3_meanpm(df_all)

    out_path = "./table3_meanpm.csv"
    table3.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"[OK] Saved: {out_path}")
    print(table3.to_string(index=False))

if __name__ == "__main__":
    main()

[OK] Saved: ./table3_meanpm.csv
Feature Set DAG Method Best Model  K_EDGE  N_FEAT AUROC (Mean ± SD) AUPRC (Mean ± SD) F1-Score (Mean ± SD) Brier (Mean ± SD)   ECE (Mean ± SD)
          O          –    XGBoost       0      13     0.936 ± 0.003     0.455 ± 0.004        0.443 ± 0.020 0.01609 ± 0.00012 0.01101 ± 0.00044
          F        GES   LightGBM      34      34     0.928 ± 0.002     0.468 ± 0.004        0.453 ± 0.007 0.01690 ± 0.00023 0.01684 ± 0.00039
          F      GOLEM   LightGBM      25      25     0.889 ± 0.005     0.350 ± 0.012        0.386 ± 0.027 0.01903 ± 0.00031 0.01880 ± 0.00077
          F    NOTEARS    XGBoost       6       6     0.867 ± 0.003     0.247 ± 0.006        0.320 ± 0.022 0.01908 ± 0.00006 0.00969 ± 0.00065
          F         PC   LightGBM      21      21     0.906 ± 0.003     0.422 ± 0.004        0.435 ± 0.012 0.01762 ± 0.00014 0.01737 ± 0.00045
         OF        GES   LightGBM      34      47     0.933 ± 0.001     0.491 ± 0.003        0.479 ± 0.017 0.0